In [2]:
import uuid, time
from src.config.parameters_config import get_notebook_parameters
from src.config.silver_config import SilverConfig
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.reader import DataFrameReader
from src.io.writer import DataFrameWriter
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

In [ ]:
layer = "silver"

#Load Spark Session
spark = get_spark_session()
dataset = get_notebook_parameters()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

silver_cfg = SilverConfig(dataset, layer)
silver_cfg.load_yaml()

In [4]:
# Create variables from yaml
target_table = silver_cfg.target_table
target_schema = silver_cfg.target_schema
target_catalog = silver_cfg.target_catalog
target_full_path = f"{target_catalog}.{target_schema}.{target_table}"
source_table = silver_cfg.source_table
source_schema = silver_cfg.source_schema
source_catalog = silver_cfg.source_catalog
pk_columns = silver_cfg.primary_keys
version_column = silver_cfg.version_column

# Create logical variables
run_id = str(uuid.uuid4())

# Create paths
checkpoint_path = f"/Volumes/{target_catalog}/landing/checkpoints/silver/{dataset}/"


In [5]:
log.info(f"[PARAMS] dataset={dataset} catalog={target_catalog} layer={layer} run_id={run_id}")
log.info(f"[CTX] target_full_path={target_full_path}")
log.info(f"[CTX] checkpoint_path={checkpoint_path}")
start = time.time()
log.info(f"[SILVER][START] run_id={run_id}")

try:
    reader = DataFrameReader(spark)
    writer = DataFrameWriter(spark)

    # Stream from Bronze (incremental via checkpoint)
    df_stream = reader.read_table_stream(
        catalog=source_catalog,
        schema=source_schema,
        table=source_table
    )
    log.info(f"[STEP 1] READ: Bronze stream loaded")

    # Apply schema + metadata (column operations work on streaming DataFrames)
    df_stream = silver_cfg.apply_schema(df_stream)
    df_stream = silver_cfg.add_metadata(df_stream, run_id)
    log.info(f"[STEP 2] TRANSFORM: Schema + metadata applied")

    # Deduplicate
    def process_batch(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        window_partition = Window.partitionBy(*pk_columns).orderBy(col(version_column).desc())
        print(window_partition)
        batch_df = (batch_df
            .withColumn("_rn", row_number().over(window_partition))
            .filter(col("_rn") == 1)
            .drop("_rn")
        )

        # MERGE into Silver (handles cross-batch dedup via UPDATE)
        writer.upsert_table(batch_df, target_full_path, pk_columns)

    writer.write_stream_with_batch(df_stream, checkpoint_path, process_batch)
    log.info(f"[STEP 3] WRITE: Stream + MERGE completed")

    duration_s = round(time.time() - start, 3)
    log.info(f"[SILVER][SUCCESS] run_id={run_id} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[SILVER][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise

ERROR:order_reviews_silver:[SILVER][FAILED] run_id=c75a85b9-49b9-451b-9734-80f8f23a927d duration_s=18.563 error=StreamingPythonRunnerInitializationException('[STREAMING_PYTHON_RUNNER_INITIALIZATION_FAILURE] Streaming Runner initialization failed, returned -2. Cause: Traceback (most recent call last):\n  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/serializers.py", line 194, in _read_with_length\n    return self.loads(obj)\n           ^^^^^^^^^^^^^^^\n  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/serializers.py", line 693, in loads\n    return cloudpickle.loads(obj, encoding=encoding)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\nModuleNotFoundError: No module named \'src\'\n\nDuring handling of the above exception, another exception occurred:\n\nTraceback (most recent call last):\n  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/sql/connect/streaming/worker/foreach_batch_worker.py", line 128, in main\n    func = 

StreamingPythonRunnerInitializationException: [STREAMING_PYTHON_RUNNER_INITIALIZATION_FAILURE] Streaming Runner initialization failed, returned -2. Cause: Traceback (most recent call last):
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/serializers.py", line 194, in _read_with_length
    return self.loads(obj)
           ^^^^^^^^^^^^^^^
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/serializers.py", line 693, in loads
    return cloudpickle.loads(obj, encoding=encoding)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'src'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/sql/connect/streaming/worker/foreach_batch_worker.py", line 128, in main
    func = worker.read_command(pickle_ser, infile)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/spark/python/_en